# Contract Intelligence — Full GPU Training on Colab T4

This notebook fine-tunes Legal-BERT on all 22,450 CUAD QA pairs using a T4 GPU.  
Expected runtime: **~2 hours** on a Colab T4.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Add your W&B API key to Colab secrets (key icon on left sidebar) named `WANDB_API_KEY`

In [ ]:
# Verify GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU found — switch runtime to T4!')

import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Install dependencies
!pip install -q transformers datasets accelerate wandb python-dotenv

In [ ]:
# Clone repo (or pull latest if already cloned)
import os

REPO_URL = 'https://github.com/Dilipchennam3005/contract-intelligence.git'
REPO_DIR = '/content/contract-intelligence'

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Set W&B API key from Colab secrets
from google.colab import userdata
import os

os.environ['WANDB_API_KEY']  = userdata.get('WANDB_API_KEY')
os.environ['WANDB_ENTITY']   = 'chennamdilip1-bur'
os.environ['WANDB_PROJECT']  = 'contract-intelligence'

print('W&B credentials set.')

# Quick login check
import wandb
wandb.login()
print('W&B login successful.')

In [ ]:
# Download CUAD dataset (~100 MB)
!python -m src.data.download_cuad

import json
from pathlib import Path

train_path = Path('data/raw/cuad_train.json')
test_path  = Path('data/raw/cuad_test.json')

with open(train_path) as f:
    train_data = json.load(f)
with open(test_path) as f:
    test_data = json.load(f)

# Count QA pairs
def count_qa(data):
    return sum(len(p['qas']) for article in data['data'] for p in article['paragraphs'])

print(f'Train QA pairs : {count_qa(train_data):,}')
print(f'Test  QA pairs : {count_qa(test_data):,}')

In [ ]:
# ── FULL GPU TRAINING ─────────────────────────────────────────────────────────
# All 22,450 training samples | max_windows=15 | 3 epochs | batch 16 | fp16
# Estimated time: ~2 hours on T4
#
# Flags:
#   --max_windows 15   : use up to 15 sliding windows per sample
#                        (None would use all ~56 but takes 6+ hrs)
#   --epochs 3         : 3 passes over the data
#   --batch_size 16    : T4 has 15 GB VRAM — 16 fits comfortably at 512 tokens

!python -m src.training.train \
    --max_windows 15 \
    --epochs 3 \
    --batch_size 16 \
    --run_name step4-finetune-full-gpu-colab

In [ ]:
# Check that the model was saved
import os
model_dir = 'models/final/step4-finetune-full-gpu-colab'
files = os.listdir(model_dir)
print(f'Files in {model_dir}:')
for f in sorted(files):
    size = os.path.getsize(f'{model_dir}/{f}') / 1e6
    print(f'  {f:<40} {size:.1f} MB')

In [ ]:
# Run proper per-clause evaluation (Step 5)
import os
os.environ['MODEL_DIR'] = 'models/final/step4-finetune-full-gpu-colab'

!python -m src.training.evaluate

In [ ]:
# Package model weights for download
import shutil, os

model_dir = 'models/final/step4-finetune-full-gpu-colab'
zip_path  = '/content/contract-intelligence-model.zip'

shutil.make_archive(
    zip_path.replace('.zip', ''),
    'zip',
    root_dir='models/final',
    base_dir='step4-finetune-full-gpu-colab'
)

size_mb = os.path.getsize(zip_path) / 1e6
print(f'Packaged model: {zip_path} ({size_mb:.0f} MB)')

In [ ]:
# Download the zip to your local machine
from google.colab import files
files.download('/content/contract-intelligence-model.zip')
print('Download started. Save the zip, then unzip into models/final/ in your local repo.')

## After downloading

1. Unzip `contract-intelligence-model.zip` into `models/final/` in your local repo
2. Update `MODEL_DIR` in your `.env` or `Dockerfile`:
   ```
   MODEL_DIR=models/final/step4-finetune-full-gpu-colab
   ```
3. Commit the new model weights and push:
   ```bash
   git add models/final/step4-finetune-full-gpu-colab/
   git commit -m 'Add GPU-trained model weights from Colab T4'
   git push
   ```
4. Rebuild Docker image: `docker compose up --build`

Check W&B for training curves: https://wandb.ai/chennamdilip1-bur/contract-intelligence